## Tiktok

#### Useful links:
- [How to set an app for Research API](https://developers.tiktok.com/products/research-api/)
- [Research API Documentation](https://developers.tiktok.com/doc/about-research-api)
- [Access Token Management](https://developers.tiktok.com/doc/client-access-token-management)

In [35]:
# requirements
%pip install pandas requests python-dotenv jmespath

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [36]:
import os
import json
import pandas as pd
import jmespath
import requests
from dotenv import load_dotenv
from datetime import datetime
from time import sleep

In [37]:
# environment variables
load_dotenv()
TIKTOK_CLIENT_KEY = os.getenv("TIKTOK_CLIENT_KEY")
TIKTOK_CLIENT_SECRET = os.getenv("TIKTOK_CLIENT_SECRET")

In [38]:
import os
from dotenv import load_dotenv

load_dotenv()  # add dotenv_path="path/to/.env" if it’s not in the CWD
TIKTOK_CLIENT_KEY = os.getenv("TIKTOK_CLIENT_KEY")
TIKTOK_CLIENT_SECRET = os.getenv("TIKTOK_CLIENT_SECRET")

print("key?", bool(TIKTOK_CLIENT_KEY), "secret?", bool(TIKTOK_CLIENT_SECRET))


key? True secret? True


In [ ]:
import os
from dotenv import load_dotenv
import requests

# .env is one level up from UK/
load_dotenv(dotenv_path="../.env", override=True)
print("key?", bool(os.getenv("TIKTOK_CLIENT_KEY")), "secret?", bool(os.getenv("TIKTOK_CLIENT_SECRET")))

url = "https://open.tiktokapis.com/v2/oauth/token/"
headers = {"Content-Type": "application/x-www-form-urlencoded", "Cache-Control": "no-cache"}
data = {
    "client_key": os.getenv("TIKTOK_CLIENT_KEY"),
    "client_secret": os.getenv("TIKTOK_CLIENT_SECRET"),
    "grant_type": "client_credentials",
}

resp = requests.post(url, headers=headers, data=data)  # <-- use data= for form
print(resp.status_code, resp.text)  # see full body if it still fails

resp_json = resp.json()
access_token = resp_json.get("access_token")
token_expiration = resp_json.get("expires_in")

print("ACCESS TOKEN:", access_token)
print("EXPIRES IN (SECONDS):", token_expiration)


#### API Access Token & How to connect

In [ ]:
from dotenv import load_dotenv
import os, requests

load_dotenv(dotenv_path="../.env", override=True)
print("key?", bool(os.getenv("TIKTOK_CLIENT_KEY")), "secret?", bool(os.getenv("TIKTOK_CLIENT_SECRET")))

url = "https://open.tiktokapis.com/v2/oauth/token/"
headers = {"Content-Type": "application/x-www-form-urlencoded", "Cache-Control": "no-cache"}
data = {
    "client_key": os.getenv("TIKTOK_CLIENT_KEY"),
    "client_secret": os.getenv("TIKTOK_CLIENT_SECRET"),
    "grant_type": "client_credentials",
}
resp = requests.post(url, headers=headers, data=data)  # <-- use data=
print(resp.status_code, resp.text)

resp_json = resp.json()
access_token = resp_json.get("access_token")
token_expiration = resp_json.get("expires_in")
print("ACCESS TOKEN:", access_token)
print("EXPIRES IN (SECONDS):", token_expiration)


In [48]:
print(bool(access_token))

True


In [41]:
# BASE URL
base_url = "https://open.tiktokapis.com/v2/research/"

In [49]:
# API requests
def request_to_api(access_token, endpoint_url, payload):
    headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}
    response = requests.post(endpoint_url, headers=headers, json=payload)
    return response.json()

# Pagination
def paginate(access_token, endpoint_url, payload, response):
    collected_data = []
    while jmespath.search("data.has_more", response):
        cursor = jmespath.search("data.cursor", response)
        search_id = jmespath.search("data.search_id", response)
        payload.update({"cursor": cursor, "search_id": search_id})                                                                  
        sleep(1) 
        response = request_to_api(access_token, endpoint_url, payload)
        data = response.get("data", [])
        collected_data.extend(data)
    return collected_data


#### Payload

In [50]:
payload_example = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["US", "CA"]},
            {"operation": "EQ", "field_name": "keyword", "field_values": ["hello world"]}
        ]
    },
    "start_date": "20250901", #format: YYYYMMDD
    "end_date": "20250910", #format: YYYYMMDD
    "max_count": 10
}

### SPECIAL CRITERIA

**SC1: Does the platform offer an API for collecting public user-generated content data?**

*This item verifies whether the platform provides an API with at least one endpoint for programmatically extracting public user-generated content to the users’ infrastructure. Public user-generated content is defined here as any publicly visible publication accessible by any platform user. The assessment should verify that the endpoint allows retrieval and storage of this content without requiring privileged or internal access beyond standard developer registration.*

In [51]:
from datetime import datetime, timedelta
if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=7)

payload = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["UK"]},
            {"operation": "EQ", "field_name": "keyword", "field_values": ["taylor swift"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

response = request_to_api(access_token, full_url, payload)

videos = response.get("data", {})
if isinstance(videos, dict):
    videos = videos.get("videos") or videos.get("data") or []
elif not isinstance(videos, list):
    videos = []

print(json.dumps(
    {
        "endpoint_details": {"method": "POST", "endpoint": endpoint_suffix},
        "fields": fields.split(","),
        "search_parameters": payload["query"]["and"],
        "sample_video": videos[:3],
        "raw_response_error": response.get("error"),
    },
    indent=2,
))

{
  "endpoint_details": {
    "method": "POST",
    "endpoint": "video/query"
  },
  "fields": [
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username"
  ],
  "search_parameters": [
    {
      "operation": "IN",
      "field_name": "region_code",
      "field_values": [
        "UK"
      ]
    },
    {
      "operation": "EQ",
      "field_name": "keyword",
      "field_values": [
        "taylor swift"
      ]
    }
  ],
  "sample_video": [],
  "raw_response_error": {
    "code": "ok",
    "message": "",
    "log_id": "20251217163631572DD5769A2E85084798"
  }
}


In [54]:
if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=7)

payload = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
            {"operation": "EQ", "field_name": "keyword", "field_values": ["Harry Potter"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

response = request_to_api(access_token, full_url, payload)

videos = response.get("data", {})
if isinstance(videos, dict):
    videos = videos.get("videos") or videos.get("data") or []
elif not isinstance(videos, list):
    videos = []

print(json.dumps(
    {
        "endpoint_details": {"method": "POST", "endpoint": endpoint_suffix},
        "fields": fields.split(","),
        "search_parameters": payload["query"]["and"],
        "sample_video": videos[:5],
        "raw_response_error": response.get("error"),
    },
    indent=2,
))

{
  "endpoint_details": {
    "method": "POST",
    "endpoint": "video/query"
  },
  "fields": [
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username"
  ],
  "search_parameters": [
    {
      "operation": "IN",
      "field_name": "region_code",
      "field_values": [
        "GB"
      ]
    },
    {
      "operation": "EQ",
      "field_name": "keyword",
      "field_values": [
        "Harry Potter"
      ]
    }
  ],
  "sample_video": [
    {
      "region_code": "GB",
      "share_count": 0,
      "username": "sidslego",
      "video_description": "HARRY POTTER Advent Calendar day 14 and we have Aragog \ud83d\udd77\ufe0f #fyp #afol #lego #harrypotter #legocollector ",
      "comment_count": 0,
      "create_time": 1765756755,
      "id": 7583867456951733526,
      "like_count": 17
    },
    {
      "comment_count": 0,
      "create_time": 1765756184,
      "id": 7583864993528007958,
    

### ACCESSIBILITY


**OC1: Does the platform offer any type of access to non-public data (e.g., private groups, not direct messages) to approved researchers?**

*This item verifies whether, beyond the retrieval and extraction of public user-generated content data, the API permits the extraction of any data from non-public content, such as posts and comments in private groups or discussion forums, subject to the implementation of adequate data hashing measures and specific researcher approval. The assessment should confirm that the API provides such access measures, either through specific endpoints or other controlled access mechanisms.*

In [ ]:
# No, the Tiktok Research API only allows access to public data.

**OC2: Can the requested data be extracted directly from the platform’s API response?**

*This item verifies whether the API returns structured data directly in its response, rather than only providing a redirect link to the data. Audiovisual media (e.g., images, videos, and audio) are excluded from this assessment. The assessment should check sample API responses to confirm that the requested public data is included in the returned payload itself.*


In [56]:
if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "view_count",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=30)

payload = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
            {"operation": "EQ", "field_name": "keyword", "field_values": ["The Incredibles"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

response = request_to_api(access_token, full_url, payload)

videos = response.get("data", {})
if isinstance(videos, dict):
    videos = videos.get("videos") or videos.get("data") or []
elif not isinstance(videos, list):
    videos = []

print(json.dumps(
    {
        "endpoint_details": {"method": "POST", "endpoint": endpoint_suffix},
        "fields": fields.split(","),
        "search_parameters": payload["query"]["and"],
        "sample_video": videos[:1],
        "raw_response_error": response.get("error"),
    },
    indent=2,
))

{
  "endpoint_details": {
    "method": "POST",
    "endpoint": "video/query"
  },
  "fields": [
    "id",
    "video_description",
    "create_time",
    "view_count",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username"
  ],
  "search_parameters": [
    {
      "operation": "IN",
      "field_name": "region_code",
      "field_values": [
        "GB"
      ]
    },
    {
      "operation": "EQ",
      "field_name": "keyword",
      "field_values": [
        "The Incredibles"
      ]
    }
  ],
  "sample_video": [
    {
      "id": 7583783359164304662,
      "region_code": "GB",
      "share_count": 0,
      "video_description": "Incredibles edit",
      "view_count": 255,
      "comment_count": 0,
      "create_time": 1765737168,
      "like_count": 14,
      "username": "jackson_henrydanger"
    }
  ],
  "raw_response_error": {
    "code": "ok",
    "message": "",
    "log_id": "20251217163948BC7670B83BFFE10819C2"
  }
}


**OC4: Does the platform’s API offer an endpoint for extracting data from an individual publication?**

*This item verifies whether it is possible to collect data from a specific public post on the platform using a unique identifier, rather than relying on search terms or other filters. The assessment should review the API documentation and test available endpoints to confirm that an individual publication can be retrieved directly by its unique identifier.*


In [58]:
# Use this cell to develop an example where a request for a specific post is made (by its id).
# Please leave the result as the output of this cell. 
post_id =  7583783359164304662 # SET THE VALUE HERE using 
endpoint_suffix = "video/query" # SET correct value (e.g "video/query")
fields = "region_code,music_id,view_count" # SET correct value (e.g "id,like_count")
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"
payload = {
    "query": {
        "and": [
            {"operation": "EQ", "field_name": "video_id", "field_values": [str(post_id)]}
        ]
    },
    "start_date": "20251115",  # widen as needed; YYYYMMDD
    "end_date":   "20251215",
    "max_count": 10
}

response = request_to_api(access_token, full_url, payload)
print(json.dumps(response, indent=2))


{
  "data": {
    "cursor": 1,
    "has_more": false,
    "search_id": "7584866126509806605",
    "videos": [
      {
        "music_id": 7583783385006623510,
        "region_code": "GB",
        "view_count": 255
      }
    ]
  },
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "202512171640493BAE231EF59E5D07E231"
  }
}


**OC5: Does the platform’s API offer an endpoint for extracting data from an individual author?**

*This item verifies whether it is possible to collect data from public posts made by a specific author, using their username or unique identifier. The assessment should review the API documentation and test relevant endpoints to confirm that data can be retrieved directly for an individual author.*


In [60]:
# Use this cell to develop an example where a request for a specific author is made.
# Please leave the result as the output of this cell.
author = "taylorswift" # SET THE VALUE HERE
endpoint_suffix = "video/query" # SET correct value (e.g "video/query")
fields = "id,video_description,view_count" # SET correct value (e.g "id,like_count")
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"


payload = {
    "query": {
        "and": [
             {"operation": "EQ", "field_name": "username", "field_values": [author]},
        ]
    },
    "start_date": "20251125",  # check other dates as well
    "end_date":   "20251205",
    "max_count": 10
}

response = request_to_api(access_token, full_url, payload)
print(json.dumps(response, indent=2))

{
  "data": {
    "search_id": "7584866126509888525",
    "videos": [
      {
        "id": 7579068769562135838,
        "video_description": "Just 11 days until the final show of The Eras Tour is all yours. The Final Show now featuring THE TORTURED POETS DEPARTMENT on @Disney+ beginning December 12 \u2728\ud83c\udf82",
        "view_count": 4315025
      }
    ],
    "cursor": 1,
    "has_more": false
  },
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "20251217170600789629589BACA20A4761"
  }
}


**OC6: Does the platform’s API provide an endpoint for extracting data based on search terms?**

*This item verifies whether public user-generated content can be collected via individual or combined search terms, enabling the creation of datasets of posts mentioning those terms. The assessment should test search-related endpoints to confirm that queries using keywords return relevant public posts.*


In [63]:
endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=7)  # adjust if you want a wider window

payload = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},  # adjust regions as needed
            {"operation": "EQ", "field_name": "keyword", "field_values": ["Santa Claus"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

response = request_to_api(access_token, full_url, payload)

videos = response.get("data", {})
if isinstance(videos, dict):
    videos = videos.get("videos") or videos.get("data") or []
elif not isinstance(videos, list):
    videos = []

print(json.dumps(
    {
        "endpoint_details": {"method": "POST", "endpoint": endpoint_suffix},
        "fields": fields.split(","),
        "search_parameters": payload["query"]["and"],
        "videos_found": len(videos),
        "sample_videos": videos[:3],  # first 3; change slice or drop it to print all
        "raw_response_error": response.get("error"),
    },
    indent=2,
))

{
  "endpoint_details": {
    "method": "POST",
    "endpoint": "video/query"
  },
  "fields": [
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username"
  ],
  "search_parameters": [
    {
      "operation": "IN",
      "field_name": "region_code",
      "field_values": [
        "GB"
      ]
    },
    {
      "operation": "EQ",
      "field_name": "keyword",
      "field_values": [
        "Santa Claus"
      ]
    }
  ],
  "videos_found": 5,
  "sample_videos": [
    {
      "like_count": 0,
      "region_code": "GB",
      "share_count": 0,
      "username": "priscillaobah3",
      "video_description": "It's Christmas Eve, and you're cozy with your loved ones, surrounded by twinkling lights and the sweet scent of freshly baked cookies. The fire crackles, and laughter fills the air. You reflect on some of your best Christmas moments. Remember decorating the tree with family, singing carols toget

**OC7: Does the API use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are returned in a locale-neutral format, or whether relevant locale metadata is included when neutrality is not possible. The assessment should review the API documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


In [65]:
endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "create_time",
    "region_code",
    "share_count",
    "view_count",
    "like_count",
    "comment_count",
    "favorites_count",
    "video_duration",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=7)  # widen/narrow as needed

payload = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
            {"operation": "EQ", "field_name": "keyword", "field_values": ["Disney"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_videos": videos[:5],  # first 5 (or fewer if not enough matches)
}, indent=2))

import re
from datetime import datetime, timezone

videos = resp.get("data", {}).get("videos", []) or []

iso8601 = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(\.\d+)?(Z|[+-]\d{2}:\d{2}|[+-]\d{4})$")
numeric_fields = ["like_count", "comment_count", "share_count", "view_count", "favorites_count"]

def to_iso(ts):
    if isinstance(ts, (int, float)):
        return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat().replace("+00:00", "Z")
    return ts

def analyze_video(v):
    ts = v.get("create_time")
    ts_iso = to_iso(ts)
    ts_is_iso = isinstance(ts_iso, str) and bool(iso8601.match(ts_iso))
    ts_is_epoch = isinstance(ts, (int, float))
    numeric_ok = all(isinstance(v.get(f), (int, float, type(None))) for f in numeric_fields)
    return {
        "id": v.get("id"),
        "create_time_raw": ts,
        "create_time_iso": ts_iso,
        "create_time_is_iso8601": ts_is_iso,
        "create_time_is_epoch": ts_is_epoch,
        "numeric_fields_are_numbers": numeric_ok,
    }

analysis = [analyze_video(v) for v in videos[:5]]

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_analysis": analysis,
}, indent=2))

{
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "20251217171204E4148C21322A4A093303"
  },
  "videos_found": 4,
  "sample_videos": [
    {
      "favorites_count": 34,
      "like_count": 580,
      "region_code": "GB",
      "view_count": 14332,
      "video_duration": 18,
      "comment_count": 2,
      "create_time": 1765756697,
      "id": 7583867191020522774,
      "share_count": 223
    },
    {
      "id": 7583867169193135362,
      "share_count": 1,
      "video_duration": 60,
      "create_time": 1765756691,
      "favorites_count": 8,
      "region_code": "GB",
      "view_count": 817,
      "comment_count": 0,
      "like_count": 61
    },
    {
      "region_code": "GB",
      "view_count": 243,
      "comment_count": 1,
      "favorites_count": 0,
      "id": 7583866987277749526,
      "like_count": 10,
      "create_time": 1765756647,
      "share_count": 0,
      "video_duration": 185
    },
    {
      "favorites_count": 25,
      "id": 7583866984656325

### COMPLIANCE

**OC15: Does the platform provide a way to label content that has been generated with artificial intelligence?**

*This item verifies whether the platform automatically flags, or allows users to flag, AI-generated content, and whether this information is given in the API response. The assessment should review the documentation and test API outputs to confirm that these flags are included in the extracted data.* 


In [74]:


from datetime import datetime, timedelta
import json

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "create_time",
    "region_code",
    "share_count",
    "view_count",
    "like_count",
    "comment_count",
    "favorites_count",
    "video_duration",
    "video_tag",  # struct array with number/type
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=30)

payload = {
    "query": {
        "and": [
             {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
             {"operation": "EQ", "field_name": "keyword", "field_values": ["photo editor"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    #"max_count": 10,
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []

# See the video_tag documentation here:
# https://developers.tiktok.com/doc/research-api-specs-query-videos

TAG_MAP = {
    ("AIGC Type", 1): "Creator labelled as AI-generated",
    ("AIGC Type", 2): "AI-generated (system)",
    ("Branded Type", 1): "Paid partnership",
    ("Branded Type", 7): "Creator earns commission",
}

def summarize_tags(tag_list):
    if not isinstance(tag_list, list):
        return []
    out = []
    for t in tag_list:
        ttype = (t or {}).get("type")
        num = (t or {}).get("number")
        label = TAG_MAP.get((ttype, num))
        out.append({"type": ttype, "number": num, "label": label})
    return out

analysis = [
    {
        "id": v.get("id"),
        "video_tag_raw": v.get("video_tag"),
        "video_tag_summary": summarize_tags(v.get("video_tag")),
    }
    for v in videos
]

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_videos": videos[:3],       # first 3 full records
    "tag_analysis": analysis[:20],     # tag summaries for up to 10
}, indent=2))


{
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "20251217171914D41B18B20FA1D609D1A1"
  },
  "videos_found": 19,
  "sample_videos": [
    {
      "region_code": "GB",
      "share_count": 1,
      "video_duration": 0,
      "view_count": 295,
      "comment_count": 0,
      "create_time": 1765745569,
      "id": 7583819447354412310,
      "favorites_count": 14,
      "like_count": 14
    },
    {
      "comment_count": 26,
      "create_time": 1765726137,
      "favorites_count": 4,
      "like_count": 158,
      "share_count": 75,
      "view_count": 213,
      "id": 7583735982969703702,
      "region_code": "GB",
      "video_duration": 12
    },
    {
      "like_count": 21,
      "region_code": "GB",
      "video_duration": 16,
      "create_time": 1765695959,
      "favorites_count": 0,
      "id": 7583606324831718678,
      "comment_count": 2,
      "share_count": 0,
      "view_count": 191
    }
  ],
  "tag_analysis": [
    {
      "id": 7583819447354412310,
   

Out of the 19 videos analysed above, 3 are labelled as AI.

### COMPLETENESS

**OC16: Can data from a publication’s comments be extracted using the platform’s API?**

*This item verifies whether comment data, including their content, can be extracted when available on the platform, either together with publication data or with a dedicated endpoint. The assessment should test relevant endpoints to confirm that comments are retrievable as structured data. This item does not apply to platforms that do not have commenting features.*


In [78]:


video_id = 7582434529894337814 # using a video with comments

endpoint_suffix = "video/comment/list/"
fields = "id,text,like_count,reply_count,parent_comment_id,create_time,video_id"
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

payload = {"video_id": video_id, "max_count": 20, "cursor": 0}
headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}

r = requests.post(full_url, headers=headers, json=payload)
print("status:", r.status_code, "ct:", r.headers.get("content-type"))
print("preview:", r.text[:400])  # see if it’s HTML/empty/error
r.raise_for_status()
data = r.json()

comments = data.get("data", {}).get("comments", []) or []
print(json.dumps({
    "error": data.get("error"),
    "comments_found": len(comments),
    "comment_texts": [c.get("text") for c in comments[:20]],
}, indent=2))


status: 200 ct: application/json
preview: {"data":{"comments":[{"video_id":7582434529894337814,"create_time":1765814081,"id":7584113677903119125,"like_count":0,"parent_comment_id":7582434529894337814,"reply_count":0,"text":"🥰🥰🥰"},{"create_time":1765426973,"id":7582451054816035591,"like_count":0,"parent_comment_id":7582434529894337814,"reply_count":0,"text":"🥰🥰🥰","video_id":7582434529894337814},{"id":7582437157491671830,"like_count":0,"par
{
  "error": {
    "code": "ok",
    "message": "ok",
    "log_id": "20251217172243B5C7CC4395CDD60988D7"
  },
  "comments_found": 3,
  "comment_texts": [
    "\ud83e\udd70\ud83e\udd70\ud83e\udd70",
    "\ud83e\udd70\ud83e\udd70\ud83e\udd70",
    "\ud83d\udc4c\ud83d\udc4c\ud83d\udc4c\ud83d\udc4c\ud83d\udc4c\ud83d\udc4c\ud83d\udc4c\ud83d\udc4c\ud83d\udc4c"
  ]
}


In [80]:
comments = data.get("data", {}).get("comments", []) or []
comment_texts = [c.get("text", "") for c in comments[:20]]

print(json.dumps({
    "error": data.get("error"),
    "comments_found": len(comments),
    "comment_texts": comment_texts,
}, indent=2, ensure_ascii=False))

{
  "error": {
    "code": "ok",
    "message": "ok",
    "log_id": "20251217172243B5C7CC4395CDD60988D7"
  },
  "comments_found": 3,
  "comment_texts": [
    "🥰🥰🥰",
    "🥰🥰🥰",
    "👌👌👌👌👌👌👌👌👌"
  ]
}


**OC17: Can data from temporary content be extracted through the platform’s API?**

*This item verifies whether the platform’s API provides at least one endpoint for collecting data from temporary publications (e.g., stories, ephemeral messages). The assessment should test endpoints to confirm whether this type of short-lived content can be retrieved as structured data before it expires. This item does not apply to platforms that do not have temporary content features.*


In [ ]:
#No endpoints or fields are provided to access temporary content such as stories. 

**OC18: Can historical data be extracted through the platform’s API?**

*This item verifies whether the API provides endpoints that allow for a specified time range, going back more than one year from the time the request is made, to collect public user-generated content data. The assessment should review test endpoints to confirm that historical data more than 12 months prior to the analysis can be retrieved.*

In [81]:
from datetime import datetime

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "share_count",
    "view_count",
    "like_count",
    "comment_count",
    "favorites_count",
    "video_duration",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

# Full year 2023
payload = {
    "query": {
        "and": [
             {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
             {"operation": "EQ", "field_name": "keyword", "field_values": ["Incredibles"]},
        ]
    },
    "start_date": "20190101",
    "end_date":   "20190131",
    "max_count": 20,  # adjust as needed
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_videos": videos[:5],  # first 5
}, indent=2))


{
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "20251217172431C585247FC7EE60098336"
  },
  "videos_found": 9,
  "sample_videos": [
    {
      "view_count": 490,
      "comment_count": 0,
      "favorites_count": 2,
      "like_count": 80,
      "share_count": 1,
      "video_description": "#duet with @iambannd #wevebeenplanning #dinner #foryoupage #frozone #frozoneswife #incredibles",
      "video_duration": 0,
      "create_time": 1548972323,
      "id": 6652785470830808325,
      "region_code": "GB"
    },
    {
      "comment_count": 0,
      "create_time": 1548926564,
      "id": 6652588935643925765,
      "like_count": 12,
      "favorites_count": 1,
      "region_code": "GB",
      "share_count": 0,
      "video_description": "#duet with @darkwolf3452 I need that suit #fluffysquad #cbduet #deadpool #incredibles",
      "video_duration": 0,
      "view_count": 471
    },
    {
      "favorites_count": 0,
      "video_description": "where's my supersuit? #incred

**OC19: Is the number of requests allowed by the API sufficient for monitoring more than 10,000 publications in 24 hours?**

*This item verifies whether data can be extracted without interruption and losses through the platform’s API for requests that accumulate more than 10,000 publications in 24 hours. The assessment should test the API to confirm that this volume of data can be collected continuously.*


In [82]:
import json, time
from datetime import datetime, timedelta
from pathlib import Path

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
stats_file = data_dir / f"eu-tiktok-UGC-question-19-stats-{timestamp}.json"

endpoint_suffix = "video/query"
fields = "id,create_time,video_description"
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

# Fixed query (adjust keyword/region if needed, keep it consistent)
payload = {
    "query": {"and": [
        {"operation": "EQ", "field_name": "region_code", "field_values": ["GB"]},
        {"operation": "EQ", "field_name": "keyword", "field_values": ["music"]}, 
    ]},
    "start_date": "20251001",
    "end_date":   "20251021",
    "max_count": 50,
}

runs = 5  # short burst for measurement; adjust if you want a longer sample
collected = 0
start = time.time()

for i in range(runs):
    resp = request_to_api(access_token, full_url, payload)
    vids = resp.get("data", {}).get("videos", []) or []
    collected += len(vids)
    print(f"run {i+1}: error={resp.get('error')} videos={len(vids)}")

elapsed = time.time() - start
reqs_per_sec = runs / elapsed if elapsed else 0
items_per_sec = collected / elapsed if elapsed else 0
items_per_24h = int(items_per_sec * 86400)

stats = {
    "runs": runs,
    "total_requests": runs,
    "total_videos_collected": collected,
    "elapsed_seconds": elapsed,
    "requests_per_second": reqs_per_sec,
    "videos_per_second": items_per_sec,
    "videos_per_24h_extrapolated": items_per_24h,
    "query": payload,
    "fields": fields.split(","),
    "endpoint": endpoint_suffix,
    "timestamp_utc": timestamp,
}

with stats_file.open("w") as fout:
    json.dump(stats, fout, indent=2)

print("\nSaved stats to", stats_file)
print(json.dumps({k: stats[k] for k in ["total_requests","total_videos_collected","requests_per_second","videos_per_24h_extrapolated"]}, indent=2))


run 1: error={'code': 'ok', 'message': '', 'log_id': '202512171724541FDB4354BED47C09FDBF'} videos=41
run 2: error={'code': 'ok', 'message': '', 'log_id': '202512171724571FDB4354BED47C09FE2A'} videos=41
run 3: error={'code': 'ok', 'message': '', 'log_id': '2025121717245912B2A9512DB73D0AE748'} videos=41
run 4: error={'code': 'ok', 'message': '', 'log_id': '20251217172500BC4F254E8DE171090716'} videos=41
run 5: error={'code': 'ok', 'message': '', 'log_id': '20251217172502BC4F254E8DE171090783'} videos=41

Saved stats to data/eu-tiktok-UGC-question-19-stats-20251217T172454Z.json
{
  "total_requests": 5,
  "total_videos_collected": 205,
  "requests_per_second": 0.5629323336465608,
  "videos_per_24h_extrapolated": 1994131
}


In [83]:
import json, time
from datetime import datetime
from pathlib import Path

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = "id,video_description,username,create_time,region_code,like_count,comment_count,share_count,view_count"
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

payload = {
    "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["climate"]},  # adjust if desired
    ]},
    "start_date": "20251001",
    "end_date":   "20251021",
    "max_count": 50,
}

#Videos per second ≈ 10.3 (890,988 per 24h ÷ 86,400).
#Time to hit 10,000 videos ≈ 10,000 ÷ 10.3 ≈ 971 seconds ≈ 16.2 minutes.

time_limit_seconds = 30  # 17 minutes based on calucations above 
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
data_file = data_dir / f"eu-tiktok-UGC-question-19-data-{ts}.jsonl"
stats_file = data_dir / f"eu-tiktok-UGC-question-19-stats-{ts}.json"

total_videos = 0
total_requests = 0
start = time.time()
cursor = None

with data_file.open("w") as fout:
    while time.time() - start < time_limit_seconds:
        if cursor:
            payload["cursor"] = cursor
        resp = request_to_api(access_token, full_url, payload)
        total_requests += 1
        vids = resp.get("data", {}).get("videos", []) or []
        for v in vids:
            json.dump(v, fout)
            fout.write("\n")
        total_videos += len(vids)
        cursor = resp.get("data", {}).get("cursor")
        has_more = resp.get("data", {}).get("has_more")
        # Stop paging if no more; optionally restart a new query to keep collecting
        if not has_more:
            cursor = None  # reset to start a fresh query
        # Minimal pacing to be gentle; adjust or remove if allowed
        time.sleep(0.5)

elapsed = time.time() - start
stats = {
    "runs": total_requests,
    "total_requests": total_requests,
    "total_videos_collected": total_videos,
    "elapsed_seconds": elapsed,
    "requests_per_second": total_requests / elapsed if elapsed else 0,
    "videos_per_second": total_videos / elapsed if elapsed else 0,
    "videos_per_24h_extrapolated": int((total_videos / elapsed) * 86400) if elapsed else 0,
    "endpoint": endpoint_suffix,
    "fields": fields.split(","),
    "query": payload.get("query"),
    "timestamp_utc": ts,
    "data_file": str(data_file),
}
with stats_file.open("w") as f:
    json.dump(stats, f, indent=2)

print("Done. Stats:", json.dumps({k: stats[k] for k in [
    "total_requests", "total_videos_collected", "requests_per_second", "videos_per_24h_extrapolated"
]}, indent=2))
print("Saved data to", data_file)
print("Saved stats to", stats_file)


Done. Stats: {
  "total_requests": 31,
  "total_videos_collected": 44,
  "requests_per_second": 1.0168910843383596,
  "videos_per_24h_extrapolated": 124703
}
Saved data to data/eu-tiktok-UGC-question-19-data-20251217T172513Z.jsonl
Saved stats to data/eu-tiktok-UGC-question-19-stats-20251217T172513Z.json


In [ ]:
# Inspect one response to see the error block and has_more
resp = request_to_api(access_token, full_url, payload)
print("error:", resp.get("error"))
print("has_more:", resp.get("data", {}).get("has_more"))
print("videos:", len(resp.get("data", {}).get("videos", []) or []))


### CONSISTENCY

**OC20: Are the results returned by the API consistently reproducible?**

*This item verifies whether data extracted via the platform’s API at any given time is consistent with other collections performed similarly, including content that has been deleted between collections. The assessment should conduct repeated test queries to confirm the reproducibility of results or ground the response based on recent (less than 2 years) experiments published in peer-reviewed journals.*


Test instructions: Please, develop a test that collects data for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [84]:
import json
from pathlib import Path
from datetime import datetime

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

# Fixed parameters for all runs
payload = {
    "query": {
        "and": [
            {"operation": "EQ", "field_name": "keyword", "field_values": ["Incredibles"]},
            {"operation": "EQ", "field_name": "region_code", "field_values": ["GB"]},
        ]
    },
    "start_date": "20230101",
    "end_date": "20230131",
    "max_count": 10,
}

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

for idx in range(5):
    resp = request_to_api(access_token, full_url, payload)
    out_path = data_dir / f"tiktok-consistency-run-{idx+1}-{timestamp}.json"
    with out_path.open("w") as fout:
        json.dump(resp, fout, indent=2)
    print(f"Saved run {idx+1} to {out_path}")


Saved run 1 to data/tiktok-consistency-run-1-20251217T172623Z.json
Saved run 2 to data/tiktok-consistency-run-2-20251217T172623Z.json
Saved run 3 to data/tiktok-consistency-run-3-20251217T172623Z.json
Saved run 4 to data/tiktok-consistency-run-4-20251217T172623Z.json
Saved run 5 to data/tiktok-consistency-run-5-20251217T172623Z.json


In [85]:
import json, glob
for path in sorted(glob.glob("data/tiktok-consistency-run-*.json")):
    with open(path) as f:
        data = json.load(f)
    vids = data.get("data", {}).get("videos", []) or []
    print(path, "videos:", len(vids), "sample_id:", vids[0].get("id") if vids else None)


data/tiktok-consistency-run-1-20251217T172623Z.json videos: 7 sample_id: 7194919220877069574
data/tiktok-consistency-run-2-20251217T172623Z.json videos: 7 sample_id: 7194919220877069574
data/tiktok-consistency-run-3-20251217T172623Z.json videos: 7 sample_id: 7194919220877069574
data/tiktok-consistency-run-4-20251217T172623Z.json videos: 7 sample_id: 7194919220877069574
data/tiktok-consistency-run-5-20251217T172623Z.json videos: 7 sample_id: 7194919220877069574


**OC21: Is the data returned by the platform’s API consistent with the parameters and filters used in the request?**

*This item verifies whether the data extracted through the API accurately reflects the parameters and filters specified in the request. The assessment should conduct repeated test queries to confirm the consistency of results or ground the response based on recent (less than 2 years) experiments published in peer-reviewed journals.*


Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [87]:

from datetime import datetime, timedelta

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "view_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

# Fixed date window
end_date = datetime.utcnow()
start_date = end_date - timedelta(days=30)

# Five different filters/parameters against the same endpoint
parameter_sets = [
    {"label": "kw_incredibles", "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["Incredibles"]},
    ]}},
    {"label": "kw_wicked", "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["Wicked"]},
    ]}},
    {"label": "kw_pixar", "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["Pixar"]},
    ]}},
    {"label": "region_GB_kw_film", "query": {"and": [
        {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
        {"operation": "EQ", "field_name": "keyword", "field_values": ["film"]},
    ]}},
    {"label": "region_GB_kw_music", "query": {"and": [
        {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
        {"operation": "EQ", "field_name": "keyword", "field_values": ["music"]},
    ]}},
]

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

def build_payload(query_block):
    return {
        "query": query_block,
        "start_date": start_date.strftime("%Y%m%d"),
        "end_date": end_date.strftime("%Y%m%d"),
        "max_count": 20,
    }

for idx, param in enumerate(parameter_sets, start=1):
    payload = build_payload(param["query"])
    resp = request_to_api(access_token, full_url, payload)
    filename = f"tiktok-UGC-question-21-run-{idx}-{param['label']}-{timestamp}.json"
    out_path = data_dir / filename
    with out_path.open("w") as fout:
        json.dump(resp, fout, indent=2)
    vids = resp.get("data", {}).get("videos", []) or []
    print(f"Saved {out_path} | videos: {len(vids)} | sample_id: {vids[0].get('id') if vids else None}")





Saved data/tiktok-UGC-question-21-run-1-kw_incredibles-20251217T172735Z.json | videos: 20 | sample_id: 7583865742966148383
Saved data/tiktok-UGC-question-21-run-2-kw_wicked-20251217T172735Z.json | videos: 16 | sample_id: 7583867620777020690
Saved data/tiktok-UGC-question-21-run-3-kw_pixar-20251217T172735Z.json | videos: 20 | sample_id: 7583867575252077845
Saved data/tiktok-UGC-question-21-run-4-region_GB_kw_film-20251217T172735Z.json | videos: 17 | sample_id: 7583867300336569623
Saved data/tiktok-UGC-question-21-run-5-region_GB_kw_music-20251217T172735Z.json | videos: 19 | sample_id: 7583867487968660758


### RELEVANCE

**OC22: Does the data extracted by the platform’s API reflect what is displayed on its user interface?**

*This item verifies whether the data returned by the API corresponds to the information displayed on the platform’s user interface at all levels of detail. The assessment should compare API responses with the user interface to confirm that key elements—such as authorship, complete content, interaction counts (e.g., comments, shares, replies), and referenced content (e.g., shares, mentions)—are fully represented.*


In [90]:


# Step 1: finding a public video id (keyword + region)
search_payload = {
    "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["Christmas"]},
        {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
    ]},
    "start_date": "20251120",
    "end_date":   "20251205",
    "max_count": 10,
}
search_resp = request_to_api(access_token, f"{base_url}video/query?fields=id", search_payload)
vids = search_resp.get("data", {}).get("videos", []) or []
print("candidate ids:", [v.get("id") for v in vids])

# Step 2: testing the id to be compared with the UI at tiktok.com/[username]/video/[video_id]
if vids:
    video_id = str(vids[0]["id"])
    by_id_payload = {
        "query": {"and": [
            {"operation": "EQ", "field_name": "video_id", "field_values": [video_id]},
        ]},
        "start_date": "20251120",
        "end_date":   "20251205",
        "max_count": 1,
    }
    resp = request_to_api(
        access_token,
        f"{base_url}video/query?fields=id,video_description,username,create_time,like_count,comment_count,share_count,view_count",
        by_id_payload,
    )
    print("error:", resp.get("error"))
    print("data:", resp.get("data"))
else:
    print("No candidates found; try widening dates/keyword or removing region filter.")


candidate ids: [7580527887317486870, 7580527877276323094, 7580527876923985174, 7580527853201116438, 7580527844388834582, 7580527842723728662, 7580527841943506198, 7580527841779977494, 7580527836931394838, 7580527835895401750]
error: {'code': 'ok', 'message': '', 'log_id': '2025121717302592624ADC6F00B50B294F'}
data: {'cursor': 1, 'has_more': False, 'search_id': '7584876084630639629', 'videos': [{'share_count': 1, 'username': 'larysa0305', 'video_description': '#fyp #tik_tok #christmas2025 #uk #❄️  @dgcomputerstore ', 'view_count': 523, 'comment_count': 10, 'create_time': 1764979196, 'id': 7580527887317486870, 'like_count': 50}]}


In [94]:
    videos = resp.get("data", {}).get("videos", []) or []
    if not videos:
        print("No video returned for", video_id)
    else:
        v = videos[0]
        print("\n--- VIDEO DETAILS ---")
        print("id:", v.get("id"))
        print("author:", v.get("username"))
        print("caption:", v.get("video_description"))
        print("create_time:", v.get("create_time"))
        print("counts -> likes:", v.get("like_count"),
              "comments:", v.get("comment_count"),
              "shares:", v.get("share_count"),
              "views:", v.get("view_count"))



--- VIDEO DETAILS ---
id: 7580527887317486870
author: larysa0305
caption: #fyp #tik_tok #christmas2025 #uk #❄️  @dgcomputerstore 
create_time: 1764979196
counts -> likes: 50 comments: 10 shares: 1 views: 523




This matches the details of this public video visible on the UI at https://www.tiktok.com/@larysa0305/video/7580527887317486870, verified on 17/12/25.

**OC23: Does the platform’s API allow for filtering data based on publisher location?**

*This item verifies whether the API supports applying location-based filters to data extraction. The assessment should test the endpoint for the main content type to confirm that data on public posts can be filtered by publisher location.*


In [96]:
endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "username",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "view_count",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=30)
keyword = "Christmas in London"  # set to None to drop keyword filter

# Build query
and_filters = [
    {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
]
if keyword:
    and_filters.append({"operation": "EQ", "field_name": "keyword", "field_values": [keyword]})

payload = {
    "query": {"and": and_filters},
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_videos": videos[:3],  # show first 3
}, indent=2))

{
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "202512171734066B8B65BB0ABA3D0A3C29"
  },
  "videos_found": 4,
  "sample_videos": [
    {
      "comment_count": 6,
      "create_time": 1765755962,
      "like_count": 30,
      "username": "reema.patel7",
      "video_description": "London  oxford Street  #christmas #london #light #travel #happiness ",
      "view_count": 411,
      "id": 7583864042884828438,
      "region_code": "GB",
      "share_count": 0
    },
    {
      "like_count": 18,
      "region_code": "GB",
      "username": "explorewithgeo",
      "view_count": 872,
      "create_time": 1765755643,
      "id": 7583862691584888086,
      "video_description": "This restaurant is so perfect for a lunch or dinner to get you in the Christmas mood!\u2728\u2764\ufe0f  It even has a set menu which looks incredible value for money\ud83c\udf77  #fyp #foodreview #christmas #londonhotspots #coventgarden ",
      "comment_count": 2,
      "share_count": 0
    },
    

**OC24: Does the platform’s API allow for filtering data based on content language?**

*This item verifies whether the API allows for applying language-based filters to data extraction. The assessment should test the endpoint for the main content type to confirm that public post data can be filtered by content language.*


In [ ]:
# This API does not support filtering data bsed on content language.


**OC25: Does the platform’s API allow for filtering data by specific time periods?**

*This item verifies whether the API allows applying temporal filters to data extraction. The assessment should test the endpoint for the main content type to confirm that public post data can be filtered by custom time ranges.*


In [101]:
from datetime import datetime, timezone
import json


def to_iso(ts):
    if isinstance(ts, (int, float)):
        return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat().replace("+00:00", "Z")
    return ts  # if it’s already a string

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "username",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "view_count",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

tests = [
    ("20241201", "20241225"),  # Dec 1–25, 2024
    ("20251101", "20251121"),  # Nov 1–21, 2025
]

for start_date, end_date in tests:
    payload = {
        "query": {"and": [
            {"operation": "EQ", "field_name": "keyword", "field_values": ["music"]},
            {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]},
        ]},
        "start_date": start_date,
        "end_date": end_date,
        "max_count": 10,
    }
    resp = request_to_api(access_token, full_url, payload)
    videos = resp.get("data", {}).get("videos", []) or []
    print(f"\nRange {start_date}–{end_date}")
    print("error:", resp.get("error"))
    print("videos_found:", len(videos))
    for v in videos[:3]:
        ct_raw = v.get("create_time")
        ct_iso = to_iso(ct_raw)
        print("  id:", v.get("id"), "| create_time:", v.get("create_time"),"| create_time_iso:", ct_iso, "| desc:", (v.get("video_description") or "")[:80])
  



Range 20241201–20241225
error: {'code': 'ok', 'message': '', 'log_id': '2025121717400964946698E8E5130BF22A'}
videos_found: 7
  id: 7452503478095596833 | create_time: 1735171190 | create_time_iso: 2024-12-25T23:59:50Z | desc: ##viral #fyp #foryou #viral #foryoupage #trending #viralvideo #parati #tiktok #x
  id: 7452503397627890976 | create_time: 1735171173 | create_time_iso: 2024-12-25T23:59:33Z | desc: duet with @dododubinks. Merry Christmas :) #producer #flstudio #musically #beat 
  id: 7452503386278219041 | create_time: 1735171167 | create_time_iso: 2024-12-25T23:59:27Z | desc: Haizz i might give up one day, im exhausted #fyp #music #sing #viral 

Range 20251101–20251121
error: {'code': 'ok', 'message': '', 'log_id': '20251217174012A848BE3D39EDC70A2A8D'}
videos_found: 10
  id: 7575332680854801686 | create_time: 1763769592 | create_time_iso: 2025-11-21T23:59:52Z | desc: Yous don’t realise what a have cooking a lot coming keep ur eyes peeled for the 
  id: 7575332643756182806 | create

### TIMELINESS

**OC26: Can data from newly published content be extracted from the platform’s API in near real time?**

*This item verifies whether the API allows the collection of data from specific content within one hour of its publication. The assessment should test the endpoint for the main content type to confirm that it allows the ready extraction of recent public posts data.*


In [103]:
endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "username",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "view_count",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=1)  # last 24h window (date-level granularity)

payload = {
    "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["Christmas"]},
        {"operation": "IN", "field_name": "region_code", "field_values": ["GB"]}
    ]},
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 20,
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []
collection_date = datetime.now(timezone.utc)

def to_iso(ts):
    if isinstance(ts, (int, float)):
        return datetime.fromtimestamp(ts, tz=timezone.utc)
    try:
        return datetime.fromisoformat(str(ts).replace("Z", "+00:00"))
    except Exception:
        return None

# Find most recent create_time
recent_dt = None
if videos:
    times = [to_iso(v.get("create_time")) for v in videos]
    times = [t for t in times if t]
    if times:
        recent_dt = max(times)

print("API error:", resp.get("error"))
print("videos_found:", len(videos))
print("Collection date (UTC):", collection_date.isoformat())

if recent_dt:
    diff = collection_date - recent_dt
    print("Most recent post (UTC):", recent_dt.isoformat())
    print("Time difference (collection - most recent):", diff)
else:
    print("Most recent post: None found (no videos or missing timestamps)")


API error: {'code': 'ok', 'message': '', 'log_id': '20251217174049C3A8D73722E6460A5941'}
videos_found: 0
Collection date (UTC): 2025-12-17T17:40:50.292442+00:00
Most recent post: None found (no videos or missing timestamps)
